# GetAround — Data Analysis & ML Pricing

**Jedha CDSD — Bloc 5 : Déploiement**

Ce notebook couvre :
1. **EDA Delay Analysis** : comprendre les retards et aider le PM à choisir le seuil minimum entre deux locations
2. **ML Pricing** : entraîner un modèle de prédiction du prix journalier

---

## 0. Setup

In [ ]:
# Installation des librairies nécessaires
!pip install openpyxl plotly scikit-learn joblib -q

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


---
# PARTIE 1 — Delay Analysis

## 1.1 Chargement des données

In [2]:
# Charger depuis Google Drive ou upload direct
# Option A — upload direct dans Colab :
# from google.colab import files
# uploaded = files.upload()

# Option B — chemin direct si fichier dans le répertoire :
df = pd.read_excel('get_around_delay_analysis.xlsx')

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

Shape : (21310, 7)
Colonnes : ['rental_id', 'car_id', 'checkin_type', 'state', 'delay_at_checkout_in_minutes', 'previous_ended_rental_id', 'time_delta_with_previous_rental_in_minutes']


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [3]:
print('--- Types ---')
print(df.dtypes)
print('\n--- Valeurs manquantes ---')
print(df.isnull().sum())
print('\n--- Stats générales ---')
df.describe()

--- Types ---
rental_id                                       int64
car_id                                          int64
checkin_type                                   object
state                                          object
delay_at_checkout_in_minutes                  float64
previous_ended_rental_id                      float64
time_delta_with_previous_rental_in_minutes    float64
dtype: object

--- Valeurs manquantes ---
rental_id                                         0
car_id                                            0
checkin_type                                      0
state                                             0
delay_at_checkout_in_minutes                   4964
previous_ended_rental_id                      19469
time_delta_with_previous_rental_in_minutes    19469
dtype: int64

--- Stats générales ---


,rental_id,car_id,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
count,21310.000000,21310.000000,16346.000000,1841.000000,1841.000000
mean,549712.880338,350030.603426,59.701517,550127.411733,279.288430
std,13863.446964,58206.249765,1002.561635,13184.023111,254.594486
min,504806.000000,159250.000000,-22433.000000,505628.000000,0.000000
25%,540613.250000,317639.000000,-36.000000,540896.000000,60.000000
50%,550350.000000,368717.000000,9.000000,550567.000000,180.000000
75%,560468.500000,394928.000000,67.000000,560823.000000,540.000000
max,576401.000000,417675.000000,71084.000000,575053.000000,720.000000


## 1.2 Distribution des types de checkin

In [4]:
checkin_counts = df['checkin_type'].value_counts().reset_index()
checkin_counts.columns = ['checkin_type', 'count']
checkin_counts['pct'] = (checkin_counts['count'] / len(df) * 100).round(1)

fig = px.pie(
    checkin_counts, values='count', names='checkin_type',
    title='Répartition des types de checkin',
    color_discrete_map={'mobile': '#636EFA', 'connect': '#EF553B'}
)
fig.show()

print(checkin_counts)

  checkin_type  count   pct
0       mobile  17003  79.8
1      connect   4307  20.2


## 1.3 Analyse des retards au checkout

> **Question PM : How often are drivers late for the next check-in?**

In [5]:
# Filtrer les locations avec une donnée de délai
df_delay = df[df['delay_at_checkout_in_minutes'].notna()].copy()

total_with_delay_data = len(df_delay)
late = df_delay[df_delay['delay_at_checkout_in_minutes'] > 0]
on_time = df_delay[df_delay['delay_at_checkout_in_minutes'] <= 0]

print(f'Locations avec données de délai : {total_with_delay_data}')
print(f'Retours en retard : {len(late)} ({len(late)/total_with_delay_data*100:.1f}%)')
print(f'Retours à l\'heure ou en avance : {len(on_time)} ({len(on_time)/total_with_delay_data*100:.1f}%)')
print(f'\nDélai médian (minutes) : {df_delay["delay_at_checkout_in_minutes"].median():.0f}')
print(f'Délai moyen (minutes) : {df_delay["delay_at_checkout_in_minutes"].mean():.0f}')

Locations avec données de délai : 16346
Retours en retard : 9404 (57.5%)
Retours à l'heure ou en avance : 6942 (42.5%)

Délai médian (minutes) : 9
Délai moyen (minutes) : 60


In [6]:
# Distribution des retards (filtrée entre -300 et 600 min pour lisibilité)
df_plot = df_delay[
    (df_delay['delay_at_checkout_in_minutes'] >= -300) &
    (df_delay['delay_at_checkout_in_minutes'] <= 600)
]

fig = px.histogram(
    df_plot, x='delay_at_checkout_in_minutes',
    color='checkin_type',
    nbins=80,
    title='Distribution des délais au checkout (en minutes)',
    labels={'delay_at_checkout_in_minutes': 'Délai (min)', 'count': 'Nombre de locations'},
    color_discrete_map={'mobile': '#636EFA', 'connect': '#EF553B'},
    barmode='overlay',
    opacity=0.7
)
fig.add_vline(x=0, line_dash='dash', line_color='black', annotation_text='Heure prévue')
fig.show()

In [7]:
# Retards par type de checkin
for ctype in ['mobile', 'connect']:
    sub = df_delay[df_delay['checkin_type'] == ctype]
    late_sub = sub[sub['delay_at_checkout_in_minutes'] > 0]
    print(f"--- {ctype.upper()} ---")
    print(f"  Total : {len(sub)}")
    print(f"  En retard : {len(late_sub)} ({len(late_sub)/len(sub)*100:.1f}%)")
    print(f"  Délai médian : {sub['delay_at_checkout_in_minutes'].median():.0f} min")
    print()

--- MOBILE ---
  Total : 12944
  En retard : 7945 (61.4%)
  Délai médian : 14 min

--- CONNECT ---
  Total : 3402
  En retard : 1459 (42.9%)
  Délai médian : -9 min



## 1.4 Impact sur la location suivante

> **Question PM : How does it impact the next driver?**

In [8]:
# Locations qui ont une location précédente
connected = df[df['previous_ended_rental_id'].notna()].copy()

print(f'Total locations : {len(df)}')
print(f'Locations avec une location précédente : {len(connected)} ({len(connected)/len(df)*100:.1f}%)')

# Récupérer le délai de la location précédente
prev_delay = df[['rental_id', 'delay_at_checkout_in_minutes']].rename(
    columns={'rental_id': 'previous_ended_rental_id',
             'delay_at_checkout_in_minutes': 'prev_driver_delay'}
)
connected = connected.merge(prev_delay, on='previous_ended_rental_id', how='left')

# Cas problématiques : le retard du conducteur précédent > temps entre deux locations
problematic = connected[
    connected['prev_driver_delay'] > connected['time_delta_with_previous_rental_in_minutes']
]

print(f'\nCas problématiques (retard précédent > temps entre locations) : {len(problematic)}')
print(f'  = {len(problematic)/len(connected)*100:.1f}% des locations avec une précédente')

Total locations : 21310
Locations avec une location précédente : 1841 (8.6%)

Cas problématiques (retard précédent > temps entre locations) : 218
  = 11.8% des locations avec une précédente


In [9]:
# Répartition des cas problématiques par type de checkin
prob_by_type = problematic['checkin_type'].value_counts().reset_index()
prob_by_type.columns = ['checkin_type', 'count']

fig = px.bar(
    prob_by_type, x='checkin_type', y='count',
    color='checkin_type',
    title='Cas problématiques par type de checkin',
    color_discrete_map={'mobile': '#636EFA', 'connect': '#EF553B'}
)
fig.show()

## 1.5 Analyse des seuils (threshold analysis)

> **Question PM : How many problematic cases will be solved depending on the threshold and scope?**
> **Question PM : How many rentals would be affected by the feature?**

In [10]:
thresholds = [0, 30, 60, 90, 120, 150, 180, 210, 240, 300, 360, 480, 720]
results = []

for scope in ['all', 'connect', 'mobile']:
    if scope == 'all':
        df_scope = connected.copy()
        df_prob = problematic.copy()
    else:
        df_scope = connected[connected['checkin_type'] == scope].copy()
        df_prob = problematic[problematic['checkin_type'] == scope].copy()

    for t in thresholds:
        solved = df_prob[df_prob['prev_driver_delay'] <= t]
        blocked = df_scope[df_scope['time_delta_with_previous_rental_in_minutes'] < t]

        results.append({
            'scope': scope,
            'threshold': t,
            'cases_solved': len(solved),
            'pct_solved': round(len(solved) / len(df_prob) * 100, 1) if len(df_prob) > 0 else 0,
            'rentals_blocked': len(blocked),
            'pct_blocked': round(len(blocked) / len(df) * 100, 1)
        })

df_results = pd.DataFrame(results)
df_results[df_results['scope'] == 'all']

,scope,threshold,cases_solved,pct_solved,rentals_blocked,pct_blocked
0,all,0,0,0.0,0,0.0
1,all,30,68,31.2,279,1.3
2,all,60,102,46.8,401,1.9
3,all,90,132,60.6,584,2.7
4,all,120,147,67.4,666,3.1
5,all,150,157,72.0,803,3.8
6,all,180,167,76.6,870,4.1
7,all,210,175,80.3,953,4.5
8,all,240,177,81.2,1001,4.7
9,all,300,189,86.7,1106,5.2


In [11]:
# Visualisation : trade-off solved vs blocked (scope = all)
df_all = df_results[df_results['scope'] == 'all']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_all['threshold'], y=df_all['pct_solved'],
    mode='lines+markers', name='% cas résolus',
    line=dict(color='green')
))

fig.add_trace(go.Scatter(
    x=df_all['threshold'], y=df_all['pct_blocked'],
    mode='lines+markers', name='% locations bloquées',
    line=dict(color='red')
))

fig.update_layout(
    title='Trade-off : cas résolus vs locations bloquées (scope = all)',
    xaxis_title='Seuil minimum (minutes)',
    yaxis_title='Pourcentage (%)',
    hovermode='x unified'
)
fig.add_vline(x=120, line_dash='dash', line_color='gray', annotation_text='120 min')
fig.show()

In [12]:
# Comparaison scope : all vs connect only
fig = px.line(
    df_results[df_results['scope'].isin(['all', 'connect'])],
    x='threshold', y='pct_solved',
    color='scope',
    title='% cas résolus : all cars vs connect only',
    labels={'threshold': 'Seuil (min)', 'pct_solved': '% cas résolus'},
    markers=True
)
fig.show()

## 1.6 Revenus impactés

> **Question PM : Which share of owner's revenue would potentially be affected?**

In [13]:
# On considère que chaque location bloquée = 1 journée de revenu perdue
# Proxy : % de locations qui seraient bloquées par le feature

# Scope all, seuils clés
key_thresholds = df_results[
    (df_results['scope'] == 'all') &
    (df_results['threshold'].isin([60, 120, 180, 240]))
][['threshold', 'rentals_blocked', 'pct_blocked', 'cases_solved', 'pct_solved']]

print('Tableau de synthèse (scope = all cars) :')
print(key_thresholds.to_string(index=False))

print('\n Recommandation : un seuil de 120 min résout 67% des cas problématiques')
print('   en ne bloquant que 3.1% des locations totales.')

Tableau de synthèse (scope = all cars) :
 threshold  rentals_blocked  pct_blocked  cases_solved  pct_solved
        60              401          1.9           102        46.8
       120              666          3.1           147        67.4
       180              870          4.1           167        76.6
       240             1001          4.7           177        81.2

 Recommandation : un seuil de 120 min résout 67% des cas problématiques
   en ne bloquant que 3.1% des locations totales.


---
# PARTIE 2 — ML Pricing Optimization

## 2.1 Chargement et exploration

In [14]:
df_price = pd.read_csv('get_around_pricing_project.csv')

# Supprimer la colonne index inutile
df_price = df_price.drop(columns=['Unnamed: 0'], errors='ignore')

print(f'Shape : {df_price.shape}')
print(f'\nValeurs manquantes : {df_price.isnull().sum().sum()}')
df_price.head()

Shape : (4843, 14)

Valeurs manquantes : 0


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [15]:
# Distribution du prix cible
fig = px.histogram(
    df_price, x='rental_price_per_day',
    title='Distribution du prix journalier (€)',
    nbins=50,
    color_discrete_sequence=['#636EFA']
)
fig.show()

print(df_price['rental_price_per_day'].describe())

count    4843.000000
mean      121.214536
std        33.568268
min        10.000000
25%       104.000000
50%       119.000000
75%       136.000000
max       422.000000
Name: rental_price_per_day, dtype: float64


In [16]:
# Prix par marque
fig = px.box(
    df_price, x='model_key', y='rental_price_per_day',
    title='Prix journalier par marque',
    color='model_key'
)
fig.update_layout(showlegend=False)
fig.show()

In [17]:
# Corrélation kilométrage / puissance avec prix
fig = px.scatter(
    df_price, x='engine_power', y='rental_price_per_day',
    color='fuel', title='Prix vs Puissance moteur',
    opacity=0.5
)
fig.show()

## 2.2 Preprocessing

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Features et target
TARGET = 'rental_price_per_day'
y = df_price[TARGET]
X = df_price.drop(columns=[TARGET])

# Colonnes catégorielles et numériques
cat_cols = ['model_key', 'fuel', 'paint_color', 'car_type']
bool_cols = ['private_parking_available', 'has_gps', 'has_air_conditioning',
             'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']
num_cols = ['mileage', 'engine_power']

# Convertir les bool en int
X[bool_cols] = X[bool_cols].astype(int)

print(f'Features : {X.shape[1]}')
print(f'Target : {TARGET}')
print(f'  min={y.min()}, max={y.max()}, mean={y.mean():.1f}')

Features : 13
Target : rental_price_per_day
  min=10, max=422, mean=121.2


In [19]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train : {X_train.shape[0]} | Test : {X_test.shape[0]}')

Train : 3874 | Test : 969


In [20]:
# Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols + bool_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# Pipeline GradientBoosting (meilleur pour régression tabulaire)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)
print('Modèle entraîné')

Modèle entraîné


## 2.3 Évaluation

In [21]:
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print(f'Train — RMSE: {rmse_train:.2f} | R²: {r2_train:.3f}')
print(f'Test  — RMSE: {rmse_test:.2f} | R²: {r2_test:.3f}')

Train — RMSE: 12.62 | R²: 0.861
Test  — RMSE: 16.02 | R²: 0.756


In [22]:
# Visualisation : prédictions vs réel
fig = px.scatter(
    x=y_test, y=y_pred_test,
    labels={'x': 'Prix réel (€)', 'y': 'Prix prédit (€)'},
    title=f'Prédictions vs Valeurs réelles — R²={r2_test:.3f}',
    opacity=0.5
)
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
              x1=y_test.max(), y1=y_test.max(),
              line=dict(color='red', dash='dash'))
fig.show()

## 2.4 Export du modèle

In [23]:
# Sauvegarder le pipeline complet
joblib.dump(pipeline, 'getaround_model.pkl')
print('Modèle exporté : getaround_model.pkl')

# Sauvegarder les noms de features pour l'API
feature_names = num_cols + bool_cols + cat_cols
joblib.dump(feature_names, 'getaround_features.pkl')
print(f'Features exportées : {feature_names}')

Modèle exporté : getaround_model.pkl
Features exportées : ['mileage', 'engine_power', 'private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires', 'model_key', 'fuel', 'paint_color', 'car_type']


In [24]:
# Test de prédiction avec un exemple
import pandas as pd

sample = pd.DataFrame([{
    'model_key': 'Renault',
    'mileage': 50000,
    'engine_power': 120,
    'fuel': 'diesel',
    'paint_color': 'grey',
    'car_type': 'sedan',
    'private_parking_available': 1,
    'has_gps': 1,
    'has_air_conditioning': 1,
    'automatic_car': 0,
    'has_getaround_connect': 1,
    'has_speed_regulator': 0,
    'winter_tires': 0
}])

pred = pipeline.predict(sample)[0]
print(f'Prédiction test : {pred:.1f} €/jour')

Prédiction test : 147.9 €/jour


---
## Résumé des résultats

### Delay Analysis
- **57.5%** des retours sont en retard (médiane : +9 min)
- **218 cas problématiques** identifiés sur 1 841 locations avec une précédente (11.8%)
- **Recommandation seuil : 120 min** → résout 67.4% des cas, bloque seulement 3.1% des locations
- **Scope : all cars** plutôt que connect only → couvre 2x plus de cas problématiques

### ML Pricing
- Modèle : GradientBoostingRegressor
- Features : kilométrage, puissance, carburant, type de voiture, équipements
- R² test : ~0.75+ — performances satisfaisantes pour la production
- Modèle exporté : `getaround_model.pkl`